# 2 · SFT (GPU job, DDP)
Fine-tunes the base model to imitate reference TL;DRs. Launches `src/sft.sh` under torchrun.

In [ ]:
import json
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('summ_config.json'))
region = cfg['region']
INSTANCE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE, image_scope='training')

env = {
    'BASE_MODEL': cfg['base_model'], 'DATA_S3': cfg['data_s3'], 'OUTPUT_SFT_S3': cfg['sft_model_s3'],
    'SEQ_LEN': str(cfg['seq_len']), 'SEED': str(cfg['seed']), 'ATTN_IMPL': cfg['attn_impl'],
    'USE_LORA': str(cfg['use_lora']), 'LORA_R': str(cfg['lora_r']), 'LORA_ALPHA': str(cfg['lora_alpha']),
    'SFT_EPOCHS': str(cfg['sft_epochs']), 'SFT_LR': str(cfg['sft_lr']),
    'SFT_BS': str(cfg['sft_bs']), 'SFT_GRAD_ACCUM': str(cfg['sft_grad_accum']),
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1', 'TOKENIZERS_PARALLELISM': 'false',
}
if cfg.get('hf_token'):
    env['HF_TOKEN'] = cfg['hf_token']; env['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='sft.sh')
ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code, compute=Compute(instance_type=INSTANCE, instance_count=1, keep_alive_period_in_seconds=cfg.get('keep_alive', 1800)),
    environment=env, base_job_name='summ-sft',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=12*3600)).train(wait=True)
print('SFT model ->', cfg['sft_model_s3'])

**Next:** `03_grpo.ipynb`.